# Capitolo 5 — Stress Test (BNN vs MLP Dropout su dati sintetici CTGAN/TVAE)

**Prerequisito**: dal notebook `Generative_final_patient_level_500_2000.ipynb`, dopo aver
eseguito tutte le celle almeno una volta, aggiungere ed eseguire in fondo la cella di export
(`export_bundle_cell.py`) — salva `stress_test_bundle.pkl` nella stessa cartella dei modelli
già pickle-ati (`model_final_bnn.pkl`, `model_final_mlp_drop.pkl`).

Questo notebook carica tutto da pickle: **non ri-allena né ri-campiona CTGAN/TVAE**, riusa
direttamente le sintesi già generate.

**Struttura (riordinata)**:
1. Setup — caricamento da pickle e ricostruzione funzioni (BNN, MLP_MC, predict, seeding)
2. Selezione programmatica della configurazione "più spread"
3. Funzioni di misura comuni ai tre livelli (aggregate + patient-level)
4. Selezione dei 10 pazienti case-study
5. Livello 1 — Perturbazione graduata (rumore gaussiano) — **tutte le analisi aggregate**
6. Livello 2 — Probe OOD estremi — **tutte le analisi aggregate**
7. Livello 3 — Permutazione (rottura di correlazione) — **tutte le analisi aggregate**
8. Metriche di sintesi (AUC di stress + sensibilità)
9. **Patient Level** — tutti i dettagli sui 10 pazienti, un livello alla volta, poi il confronto cross-livello
10. Export tabelle per LaTeX
11. Note di sintesi per il Cap.5

> Nota di riordino: le sezioni 5-7 contengono ora **solo** le analisi aggregate di ciascun
> livello, così da poterle leggere una dopo l'altra senza interruzioni. Tutto ciò che riguarda
> i 10 pazienti case-study (traiettorie, probe, permutazioni, confronto cross-livello) è stato
> raggruppato nella Sezione 9, organizzato per livello. Alcune celle di codice disattivato
> (versioni vecchie di grafici, già sostituite da versioni più recenti nello stesso notebook)
> sono state rimosse: i dettagli sono in fondo, nella cella di note.


## 1. Setup — caricamento da pickle

Le classi `BNN` e `MLP_MC` vanno ridefinite qui **identiche** a quelle del notebook
autoritativo, perché `pickle` ha bisogno della definizione della classe nel namespace
per deserializzare gli oggetti modello (altrimenti `pickle.load` solleva
`AttributeError: Can't get attribute 'BNN'...`).


In [ ]:

import os
import random
import pickle

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from blitz.modules import BayesianLinear

MODELS_DIR = "../Generative AI"  
BUNDLE_DIR = "."                 


class BNN(nn.Module):
    def __init__(self, n_features, n_outputs, M):
        super().__init__()
        self.fc1 = BayesianLinear(n_features, M)
        self.fc2 = BayesianLinear(M, M)
        self.fc3 = BayesianLinear(M, n_outputs)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        return self.fc3(x)


class MLP_MC(nn.Module):
    def __init__(self, n_features, n_outputs, M, drop_rate=0.2):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(n_features, M),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(M, M),
            nn.ReLU(),
            nn.Dropout(drop_rate),
            nn.Linear(M, n_outputs)
        )

    def forward(self, x):
        return self.model(x)

    def drop_enable(self):
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()


In [2]:

def set_all_seeds(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if hasattr(torch.backends, "cudnn"):
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True


def bnn_predict(model, x_tensor, n_samples=100, seed=None):
    if seed is not None:
        set_all_seeds(seed)
    model.eval()
    with torch.no_grad():
        preds = torch.stack([torch.sigmoid(model(x_tensor)) for _ in range(n_samples)])
    return preds.mean(0).numpy(), preds.std(0).numpy()


def mlp_mc_predict(model, x_tensor, n_samples=100, seed=None):
    if seed is not None:
        set_all_seeds(seed)
    model.eval()
    model.drop_enable()
    with torch.no_grad():
        preds = torch.stack([torch.sigmoid(model(x_tensor)) for _ in range(n_samples)])
    return preds.mean(0).numpy(), preds.std(0).numpy()


def zscore(series):
    std = series.std(ddof=0)
    if std == 0 or pd.isna(std):
        return pd.Series(0.0, index=series.index)
    return (series - series.mean()) / std


In [ ]:

with open(f"{MODELS_DIR}/model_final_bnn.pkl", "rb") as f:
    bnn_model = pickle.load(f)
with open(f"{MODELS_DIR}/model_final_mlp_drop.pkl", "rb") as f:
    mlp_model = pickle.load(f)

with open(f"{BUNDLE_DIR}/stress_test_bundle.pkl", "rb") as f:
    bundle = pickle.load(f)

GLOBAL_SEED = bundle["GLOBAL_SEED"]
feature_cols = bundle["feature_cols"]
label_cols = bundle["label_cols"]
thr_bnn = bundle["thr_bnn"]
thr_mlp_dropout = bundle["thr_mlp_dropout"]
seed_offset = bundle["seed_offset"]
sample_size = bundle["sample_size"]
scaler = bundle["scaler"]
spread_df = bundle["spread_df"]
synthetic_ctgan = bundle["synthetic_ctgan"]
synthetic_tvae = bundle["synthetic_tvae"]
data = bundle["data"]  # X_bal, real training data
patient_level_synthetic = bundle.get("patient_level_synthetic")

set_all_seeds(GLOBAL_SEED)
print(f"GLOBAL_SEED = {GLOBAL_SEED}")
print(f"label_cols = {label_cols}")
print(f"sample_size disponibili = {sample_size}")


GLOBAL_SEED = 123
label_cols = ['diabetes', 'stroke', 'HA', 'cancer']
sample_size disponibili = [500, 1000, 2000, 5000]


In [4]:

most_spread_per_generator = (
    spread_df.loc[spread_df.groupby('model')['spread_score'].idxmax()]
    [['model', 'n_samples', 'spread_score']]
    .reset_index(drop=True)
)
print("Configurazione più spread per generatore (ricalcolata da spread_score):")
print(most_spread_per_generator)

STRESS_CONFIG = {
    row['model'].lower(): int(row['n_samples'])
    for _, row in most_spread_per_generator.iterrows()
}
print("STRESS_CONFIG =", STRESS_CONFIG)


Configurazione più spread per generatore (ricalcolata da spread_score):
   model  n_samples  spread_score
0  CTGAN       1000      0.728527
1   TVAE       1000      0.502563
STRESS_CONFIG = {'ctgan': 1000, 'tvae': 1000}


In [5]:

CANDIDATE_CONTINUOUS = ["age", "bmi", "maxgrip", "chol", "crp", "trigl", "cyst_c", "hdl", "thb", "a1c"]
CONTINUOUS_FEATURES = [f for f in CANDIDATE_CONTINUOUS if f in feature_cols]
missing = set(CANDIDATE_CONTINUOUS) - set(CONTINUOUS_FEATURES)
if missing:
    print("Attenzione, non trovate in feature_cols:", missing)
print("Feature continue usate per lo stress test:", CONTINUOUS_FEATURES)

REF_STD = data[CONTINUOUS_FEATURES].std()
REF_RANGE = {f: (data[f].min(), data[f].max()) for f in CONTINUOUS_FEATURES}

gen_colors = {'ctgan': '#4C78A8', 'tvae': '#F58518'}



Feature continue usate per lo stress test: ['age', 'bmi', 'maxgrip', 'chol', 'crp', 'trigl', 'cyst_c', 'hdl', 'thb', 'a1c']


In [6]:

def predict_on_dataframe(df_input, n_mc_samples=100, seed_bnn=None, seed_mlp=None):
    x_t = torch.tensor(scaler.transform(df_input[feature_cols]), dtype=torch.float32)
    prob_bnn, std_bnn = bnn_predict(bnn_model, x_t, n_samples=n_mc_samples, seed=seed_bnn)
    prob_mlp, std_mlp = mlp_mc_predict(mlp_model, x_t, n_samples=n_mc_samples, seed=seed_mlp)
    return prob_bnn, std_bnn, prob_mlp, std_mlp


def summarize_stress_level(prob_bnn, std_bnn, prob_mlp, std_mlp, label_cols):
    eps = 1e-8
    rows = {}
    for i, col in enumerate(label_cols):
        p_bnn, p_mlp = prob_bnn[:, i], prob_mlp[:, i]
        amb_bnn = -p_bnn * np.log2(p_bnn + eps) - (1 - p_bnn) * np.log2(1 - p_bnn + eps)
        amb_mlp = -p_mlp * np.log2(p_mlp + eps) - (1 - p_mlp) * np.log2(1 - p_mlp + eps)
        rows[col] = {
            'uncertainty_bnn': std_bnn[:, i].mean() * 100,
            'ambiguity_bnn': amb_bnn.mean(),
            'uncertainty_mlp': std_mlp[:, i].mean() * 100,
            'ambiguity_mlp': amb_mlp.mean(),
            'disagreement': np.abs(p_bnn - p_mlp).mean(),
        }
    return pd.DataFrame(rows).T


In [7]:
def summarize_stress_level_patients(prob_bnn, std_bnn, prob_mlp, std_mlp, label_cols, patient_ids):
    """Per-patient/label summary instead of an aggregate over the whole dataframe.
    
    Here patient_id is treated as the row position in the dataframe passed to
    predict_on_dataframe, consistent with the patient-level bundle of Chapter 4.
    """
    eps = 1e-8
    rows = []
    n_rows = prob_bnn.shape[0]
    for pid in patient_ids:
        row_pos = int(pid)
        if row_pos < 0 or row_pos >= n_rows:
            continue
        for j, col in enumerate(label_cols):
            p_bnn, p_mlp = prob_bnn[row_pos, j], prob_mlp[row_pos, j]
            amb_bnn = -p_bnn * np.log2(p_bnn + eps) - (1 - p_bnn) * np.log2(1 - p_bnn + eps)
            amb_mlp = -p_mlp * np.log2(p_mlp + eps) - (1 - p_mlp) * np.log2(1 - p_mlp + eps)
            rows.append({
                'patient_id': row_pos, 'label': col,
                'probability_bnn': p_bnn,
                'uncertainty_bnn': std_bnn[row_pos, j] * 100,
                'ambiguity_bnn': amb_bnn,
                'probability_mlp': p_mlp,
                'uncertainty_mlp': std_mlp[row_pos, j] * 100,
                'ambiguity_mlp': amb_mlp,
            })
    return pd.DataFrame(rows)


### Patient subsample - case-study




In [ ]:
assert patient_level_synthetic is not None, (
    "patient_level_synthetic non presente nel bundle: aggiungerlo a stress_test_bundle "
    "nella cella di export del notebook autoritativo"
)

N_CASE_STUDY_PATIENTS = 10
REFERENCE_PATIENT_ID = 604  


def select_case_study_ranking(generator_key):
    n = STRESS_CONFIG[generator_key]
    df_bnn = patient_level_synthetic[(n, generator_key, 'bnn')]
    df_mlp = patient_level_synthetic[(n, generator_key, 'mlp_mc')]

    score_parts = []
    for df_tmp, tag in [(df_bnn, 'bnn'), (df_mlp, 'mlp')]:
        s = (
            zscore(df_tmp['uncertainty_pct'])
            + zscore(df_tmp['ambiguity'])
            + zscore(-df_tmp['distance_from_threshold'])
        )
        score_parts.append(
            df_tmp.assign(row_interest=s).groupby('patient_id')['row_interest'].mean().rename(tag)
        )

    merged = df_bnn.merge(df_mlp, on=['patient_id', 'label'], suffixes=('_bnn', '_mlp'))
    merged['prob_gap'] = (merged['probability_bnn'] - merged['probability_mlp']).abs()
    disagreement_interest = (
        merged.assign(gap_z=zscore(merged['prob_gap']))
        .groupby('patient_id')['gap_z'].mean()
    )
    score_parts.append(disagreement_interest.rename('disagreement'))

    interest = pd.concat(score_parts, axis=1)
    interest['interesting_score'] = interest.mean(axis=1)
    return interest['interesting_score'].sort_values(ascending=False)


def valid_patient_ids_for_generator(generator_key):
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    return set(int(i) for i in synth_df.index)


case_study_generator = list(STRESS_CONFIG.keys())[0]
case_study_ranking = select_case_study_ranking(case_study_generator)

valid_ctgan = valid_patient_ids_for_generator('ctgan')
valid_tvae = valid_patient_ids_for_generator('tvae')
valid_patient_ids = valid_ctgan & valid_tvae

CASE_STUDY_PATIENT_IDS = [
    int(pid) for pid in case_study_ranking.index
    if int(pid) in valid_patient_ids
][:N_CASE_STUDY_PATIENTS]

if REFERENCE_PATIENT_ID in valid_patient_ids and REFERENCE_PATIENT_ID not in CASE_STUDY_PATIENT_IDS:
    CASE_STUDY_PATIENT_IDS = [REFERENCE_PATIENT_ID] + CASE_STUDY_PATIENT_IDS[:N_CASE_STUDY_PATIENTS - 1]

_dropped = set(int(pid) for pid in case_study_ranking.index[:N_CASE_STUDY_PATIENTS * 2]) - valid_patient_ids
if _dropped:
    print(f"ATTENZIONE: {len(_dropped)} pazienti tra i più interessanti non sono validi in "
          f"entrambi i generatori (ctgan n={STRESS_CONFIG['ctgan']}, tvae n={STRESS_CONFIG['tvae']}) "
          f"e sono stati esclusi: {sorted(_dropped)}")

CASE_STUDY_PATIENT_ID = CASE_STUDY_PATIENT_IDS[0]
print(f"Pazienti selezionati per il case study (validi su ctgan E tvae): {CASE_STUDY_PATIENT_IDS}")
case_study_ranking.head(N_CASE_STUDY_PATIENTS)


Pazienti selezionati per il case study (validi su ctgan E tvae): [604, 492, 456, 28, 115, 922, 985, 152, 362, 378]


patient_id
492    2.069245
456    1.842315
28     1.812577
115    1.811733
922    1.722911
985    1.665857
152    1.652579
362    1.633075
378    1.628202
945    1.620014
Name: interesting_score, dtype: float64

## 5. LEVEL 1 - NOISE INJECTION


In [ ]:


SIGMA_LEVELS = [0.0, 0.05, 0.10, 0.20, 0.35, 0.50, 0.75, 1.00]


def inject_gaussian_noise(df, features, sigma_level, seed_offset_extra=0, clip_to_ref_range=False):
    """Additive Gaussian noise, scaled on REF_STD (standard deviation of the
    features on the REAL data, not on the perturbed synthetic data), so that
    sigma_level is a common unit across CTGAN and TVAE, regardless of how
    dispersed each generator already is (see Chapter 4 / spread_df).
    
    clip_to_ref_range: if True, perturbed values are clipped to REF_RANGE
    (min/max observed on the real data). Useful at high sigma to separate
    this level ("plausible" stress) from a true OOD probe (Level 2), where
    leaving the range is intentional and systematic.
    """
    rng = np.random.default_rng(GLOBAL_SEED + 50_000 + seed_offset_extra)
    df_noisy = df.copy()
    for f in features:
        noise = rng.normal(loc=0.0, scale=sigma_level * REF_STD[f], size=len(df))
        df_noisy[f] = df_noisy[f] + noise
        if clip_to_ref_range:
            lo, hi = REF_RANGE[f]
            df_noisy[f] = df_noisy[f].clip(lo, hi)
    return df_noisy


def run_level1_aggregate(generator_key, sigma_levels=None, clip_to_ref_range=False):
    sigma_levels = sigma_levels if sigma_levels is not None else SIGMA_LEVELS
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    seed_base = seed_offset[generator_key] + n
    records = []
    for k, sigma in enumerate(sigma_levels):
        df_pert = inject_gaussian_noise(synth_df, CONTINUOUS_FEATURES, sigma,
                                         seed_offset_extra=k, clip_to_ref_range=clip_to_ref_range)
        prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
            df_pert,
            seed_bnn=GLOBAL_SEED + 30_000 + seed_base + k,
            seed_mlp=GLOBAL_SEED + 40_000 + seed_base + k,
        )
        summary = summarize_stress_level(prob_bnn, std_bnn, prob_mlp, std_mlp, label_cols)
        summary['sigma_level'] = sigma
        summary['generator'] = generator_key
        summary['label'] = summary.index
        records.append(summary.reset_index(drop=True))
    return pd.concat(records, ignore_index=True)


def run_level1_per_feature(generator_key, sigma_fixed=0.20):
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    seed_base = seed_offset[generator_key] + n
    records = []
    for k, f in enumerate(CONTINUOUS_FEATURES):
        df_pert = inject_gaussian_noise(synth_df, [f], sigma_fixed, seed_offset_extra=100 + k)
        prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
            df_pert,
            seed_bnn=GLOBAL_SEED + 31_000 + seed_base + k,
            seed_mlp=GLOBAL_SEED + 41_000 + seed_base + k,
        )
        summary = summarize_stress_level(prob_bnn, std_bnn, prob_mlp, std_mlp, label_cols)
        summary['feature_perturbed'] = f
        summary['generator'] = generator_key
        summary['label'] = summary.index
        records.append(summary.reset_index(drop=True))
    return pd.concat(records, ignore_index=True)


def sigma_plausibility_check(generator_key, sigma_levels=None, features=None, seed_offset_extra_base=0):
    """For each sigma and each continuous feature, the % of perturbed values that
    fall outside REF_RANGE (range observed on the REAL data). It answers the
    question: "at which sigma does Level 1 start to conceptually overlap with
    Level 2 (extreme OOD probes)?" Without this check, what happens at high
    sigma cannot be interpreted reliably.
    """
    sigma_levels = sigma_levels if sigma_levels is not None else SIGMA_LEVELS
    features = features or CONTINUOUS_FEATURES
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    records = []
    for k, sigma in enumerate(sigma_levels):
        df_pert = inject_gaussian_noise(synth_df, features, sigma,
                                         seed_offset_extra=seed_offset_extra_base + k)
        for f in features:
            lo, hi = REF_RANGE[f]
            pct_out = float(((df_pert[f] < lo) | (df_pert[f] > hi)).mean() * 100)
            records.append({
                'generator': generator_key, 'sigma_level': sigma, 'feature': f,
                'pct_out_of_range': pct_out,
            })
    return pd.DataFrame(records)


In [ ]:

level1_aggregate = pd.concat(
    [run_level1_aggregate('ctgan'), run_level1_aggregate('tvae')], ignore_index=True
)
level1_per_feature = pd.concat(
    [run_level1_per_feature('ctgan'), run_level1_per_feature('tvae')], ignore_index=True
)

level1_aggregate_clipped = pd.concat(
    [run_level1_aggregate('ctgan', clip_to_ref_range=True),
     run_level1_aggregate('tvae', clip_to_ref_range=True)], ignore_index=True
)

plausibility_df = pd.concat(
    [sigma_plausibility_check('ctgan'), sigma_plausibility_check('tvae')], ignore_index=True
)

level1_aggregate.head()


,uncertainty_bnn,ambiguity_bnn,uncertainty_mlp,ambiguity_mlp,disagreement,sigma_level,generator,label
0,0.789857,0.736675,5.913999,0.814205,0.065061,0.00,ctgan,diabetes
1,1.168430,0.701853,5.872777,0.782933,0.086457,0.00,ctgan,stroke
2,0.762837,0.838899,4.797501,0.870245,0.057874,0.00,ctgan,HA
3,1.347029,0.725646,6.791291,0.837046,0.105549,0.00,ctgan,cancer
4,0.797079,0.735469,5.992902,0.815043,0.066074,0.05,ctgan,diabetes


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial"],
    "axes.edgecolor": "#4a4a4a",
    "axes.labelcolor": "#2b2b2b",
    "text.color": "#2b2b2b",
    "xtick.color": "#4a4a4a",
    "ytick.color": "#4a4a4a",
})

gen_colors = {
    "ctgan": "#1f4e79",   
    "tvae": "#e07b39",    
}

OUT_OF_RANGE_THRESHOLD = 5.0 


def sigma_breach_point(generator_key, threshold=OUT_OF_RANGE_THRESHOLD):
    sub = (
        plausibility_df[plausibility_df["generator"] == generator_key]
        .groupby("sigma_level")["pct_out_of_range"]
        .mean()
    )
    breach = sub[sub > threshold]
    return float(breach.index.min()) if len(breach) else None


def plot_level1_curve(df, metric_prefix, ylabel, title=None, filename=None, show_breach=True):
    labels_to_plot = list(label_cols)
    n_rows = len(labels_to_plot)

    fig, axes = plt.subplots(
        n_rows, 2, figsize=(11.5, 3.0 * n_rows), sharex=True,
    )
    if n_rows == 1:
        axes = axes.reshape(1, 2)

    breach_points = {gen: sigma_breach_point(gen) for gen in ["ctgan", "tvae"]}
    model_titles = {"bnn": "BNN", "mlp": "MLP Dropout"}

    for row, label in enumerate(labels_to_plot):
        for col_idx, model in enumerate(["bnn", "mlp"]):
            ax = axes[row, col_idx]

            for gen in ["ctgan", "tvae"]:
                sub = (
                    df[(df["generator"] == gen) & (df["label"] == label)]
                    .sort_values("sigma_level")
                )
                ax.plot(
                    sub["sigma_level"], sub[f"{metric_prefix}_{model}"],
                    marker="o", markersize=5, linewidth=1.8,
                    color=gen_colors[gen], label=gen.upper(),
                    solid_capstyle="round", zorder=3,
                )
                bp = breach_points[gen]
                if show_breach and bp is not None:
                    ax.axvline(
                        bp, color=gen_colors[gen], linestyle="--",
                        alpha=0.35, linewidth=1.1, zorder=1,
                    )

            
            ax.set_title(
                f"{label} — {model_titles[model]}",
                fontsize=10.5, fontweight="bold", pad=8,
            )

            if row == n_rows - 1:
                ax.set_xlabel("sigma_level", fontsize=9.5)
            if col_idx == 0:
                ax.set_ylabel(ylabel, fontsize=9.5)

            for spine in ["top", "right"]:
                ax.spines[spine].set_visible(False)
            ax.spines["left"].set_color("#8c8c8c")
            ax.spines["bottom"].set_color("#8c8c8c")

            ax.grid(alpha=0.25, zorder=0)
            ax.set_axisbelow(True)
            ax.tick_params(axis="both", length=0, labelsize=8.5)
            ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=5))

    handles, labels_legend = axes[0, 0].get_legend_handles_labels()
    fig.legend(
        handles, labels_legend, loc="upper center", ncol=2, frameon=False,
        bbox_to_anchor=(0.5, 1.015), fontsize=10.5,
    )

    if title:
        fig.suptitle(title, fontsize=13, fontweight="bold", y=1.035)

    fig.tight_layout(rect=[0, 0, 1, 0.97])

    if filename:
        plt.savefig(filename, dpi=150, bbox_inches="tight")
    return fig


breach_points = {gen: sigma_breach_point(gen) for gen in ["ctgan", "tvae"]}
print(
    "Sigma level beyond which >%.0f%% of values fall outside REF_RANGE "
    "(averaged across features):" % OUT_OF_RANGE_THRESHOLD
)
print(breach_points)

fig_unc = plot_level1_curve(
    level1_aggregate, "uncertainty", "Uncertainty",
    filename="level1_aggregate_uncertainty_curve.png",
)
plt.show()

fig_amb = plot_level1_curve(
    level1_aggregate, "ambiguity", "Ambiguity (%)",
    filename="level1_aggregate_ambiguity_curve.png",
)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap


plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial"],
    "axes.edgecolor": "#4a4a4a",
    "axes.labelcolor": "#2b2b2b",
    "text.color": "#2b2b2b",
    "xtick.color": "#4a4a4a",
    "ytick.color": "#4a4a4a",
})

heat_cmap = LinearSegmentedColormap.from_list(
    "soft_reds", ["#fbf4f2", "#f2b6a3", "#d9583a", "#7a1a10"]
)

plaus_pivot_ctgan = plausibility_df[plausibility_df["generator"] == "ctgan"].pivot(
    index="feature", columns="sigma_level", values="pct_out_of_range"
)
plaus_pivot_tvae = plausibility_df[plausibility_df["generator"] == "tvae"].pivot(
    index="feature", columns="sigma_level", values="pct_out_of_range"
)


data_max = max(plaus_pivot_ctgan.values.max(), plaus_pivot_tvae.values.max())
vmax = max(data_max, 1) 

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))

for ax, pivot, title in zip(axes, [plaus_pivot_ctgan, plaus_pivot_tvae], ["CTGAN", "TVAE"]):
    im = ax.imshow(pivot.values, aspect="auto", cmap=heat_cmap, vmin=0, vmax=vmax)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{s:g}" for s in pivot.columns], fontsize=9.5)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9.5)
    ax.set_xlabel("sigma_level", fontsize=10.5)
    ax.set_title(f"Values Outside REF_RANGE for {title}", fontsize=12.5, fontweight="bold", pad=10)

   
    ax.set_xticks(np.arange(-0.5, pivot.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, pivot.shape[0], 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", length=0)
    ax.tick_params(which="major", length=0)
    for spine in ax.spines.values():
        spine.set_visible(False)

    
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if val > 0:
                ax.text(
                    j, i, f"{val:.0f}", ha="center", va="center",
                    color="white" if val > vmax * 0.5 else "#4a1a10",
                    fontsize=9, fontweight="bold",
                )

cbar = fig.colorbar(im, ax=axes, shrink=0.85, pad=0.02)
cbar.set_label("% outside reference range", fontsize=10.5)
cbar.outline.set_visible(False)


plt.savefig("plausibility_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

plausibility_df.to_csv("plausibility_check.csv", index=False)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker


plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Arial"],
    "axes.edgecolor": "#4a4a4a",
    "axes.labelcolor": "#2b2b2b",
    "text.color": "#2b2b2b",
    "xtick.color": "#4a4a4a",
    "ytick.color": "#4a4a4a",
})

gen_colors = {
    "ctgan": "#1f4e79",   
    "tvae": "#e07b39",    
}

x = np.arange(len(CONTINUOUS_FEATURES))
width = 0.36

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=False)

model_specs = [
    ("uncertainty_bnn", "BNN"),
    ("uncertainty_mlp", "MLP Dropout"),
]

for ax, (model_col, title) in zip(axes, model_specs):
    agg = (
        level1_per_feature.groupby(["generator", "feature_perturbed"])[model_col]
        .mean()
        .unstack("generator")
        .reindex(CONTINUOUS_FEATURES)
    )

    bars_ctgan = ax.bar(
        x - width / 2, agg["ctgan"], width,
        label="CTGAN", color=gen_colors["ctgan"],
        edgecolor="white", linewidth=0.6, zorder=3,
    )
    bars_tvae = ax.bar(
        x + width / 2, agg["tvae"], width,
        label="TVAE", color=gen_colors["tvae"],
        edgecolor="white", linewidth=0.6, zorder=3,
    )

    ax.set_title(title, fontsize=13, fontweight="bold", pad=12)

    ax.set_xticks(x)
    ax.set_xticklabels(CONTINUOUS_FEATURES, rotation=40, ha="right", fontsize=9.5)
    ax.set_ylabel("Mean uncertainty", fontsize=10.5)

    for spine in ["top", "right", "left"]:
        ax.spines[spine].set_visible(False)
    ax.spines["bottom"].set_color("#8c8c8c")

    ax.yaxis.grid(True, alpha=0.25, zorder=0)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", length=0)

    ymax = max(agg["ctgan"].max(), agg["tvae"].max())
    ax.set_ylim(0, ymax * 1.15)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6))

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles, labels, loc="upper center", ncol=2, frameon=False,
    bbox_to_anchor=(0.5, 1.04), fontsize=11,
)

fig.suptitle(
    "Average Uncertainty per Perturbed Feature",
    fontsize=15, fontweight="bold", y=1.12,
)

plt.tight_layout()
plt.savefig("level1_per_feature.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. LEVEL 2 - OOD PROBE

In [14]:
def build_extreme_probes(generator_key, features=None, n_patients=20, margin_frac=0.15):
    """OOD probe: a globally anomalous patient.
    
    All continuous features are shifted together to
    train_max + margin_frac * range. This is the version consistent with the
    Level 2 narrative: not an isolated anomaly, but a global out-of-distribution
    profile.
    """
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    features = features or CONTINUOUS_FEATURES

    rng = np.random.default_rng(GLOBAL_SEED + 60_000)
    patient_idx = rng.choice(synth_df.index, size=min(n_patients, len(synth_df)), replace=False)

    probes = synth_df.loc[patient_idx].copy()
    for f in features:
        train_min, train_max = REF_RANGE[f]
        margin = margin_frac * (train_max - train_min)
        probes[f] = train_max + margin
    probes['_source_patient'] = patient_idx
    return probes.reset_index(drop=True)


def run_level2(generator_key, n_patients=20, margin_frac=0.15):
    n = STRESS_CONFIG[generator_key]
    seed_base = seed_offset[generator_key] + n
    probes = build_extreme_probes(generator_key, n_patients=n_patients, margin_frac=margin_frac)
    prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
        probes,
        seed_bnn=GLOBAL_SEED + 32_000 + seed_base,
        seed_mlp=GLOBAL_SEED + 42_000 + seed_base,
    )
    summary = summarize_stress_level(prob_bnn, std_bnn, prob_mlp, std_mlp, label_cols)
    summary['generator'] = generator_key
    summary['label'] = summary.index
    return summary.reset_index(drop=True), probes


In [ ]:

level2_ctgan, probes_ctgan = run_level2('ctgan')
level2_tvae, probes_tvae = run_level2('tvae')
level2_summary = pd.concat([level2_ctgan, level2_tvae], ignore_index=True)
level2_summary


In [ ]:
baseline_l1 = level1_aggregate[level1_aggregate['sigma_level'] == 0.0]

compare_records = []
for gen in ['ctgan', 'tvae']:
    for lab in label_cols:
        base_row = baseline_l1[(baseline_l1['generator'] == gen) & (baseline_l1['label'] == lab)].iloc[0]
        probe_row = level2_summary[(level2_summary['generator'] == gen) & (level2_summary['label'] == lab)].iloc[0]
        for model in ['bnn', 'mlp']:
            for metric in ['uncertainty', 'ambiguity']:
                compare_records.append({
                    'generator': gen,
                    'label': lab,
                    'model': model.upper(),
                    'metric': metric,
                    'baseline': base_row[f'{metric}_{model}'],
                    'ood_probe': probe_row[f'{metric}_{model}'],
                    'delta': probe_row[f'{metric}_{model}'] - base_row[f'{metric}_{model}'],
                })

level2_vs_baseline = pd.DataFrame(compare_records)
level2_vs_baseline


In [ ]:
def plot_level2_vs_baseline(metric, ylabel, filename=None, dpi=300):
    """
    Plot Level 2 stress-test results (baseline vs OOD probe) for BNN and MLP,
    split by synthetic data generator (CTGAN / TVAE).

    Parameters
    ----------
    metric : str
        Column value to filter on ('uncertainty' or 'ambiguity').
    ylabel : str
        Y-axis label.
    filename : str, optional
        If given, saves the figure to this path.
    dpi : int
        Resolution for the saved figure (default 300, publication-quality).
    """
    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    x = np.arange(len(label_cols))
    width = 0.2
    hatches = {'baseline': '', 'ood_probe': '//'}

    handles, labels = [], []
    for ax, model in zip(axes, ['BNN', 'MLP']):
        sub = level2_vs_baseline[
            (level2_vs_baseline['model'] == model) &
            (level2_vs_baseline['metric'] == metric)
        ]
        for i, gen in enumerate(['ctgan', 'tvae']):
            s = sub[sub['generator'] == gen].set_index('label').reindex(label_cols)
            offset_base = (i * 2) * width - width * 1.5
            offset_probe = (i * 2 + 1) * width - width * 1.5

            bar_base = ax.bar(x + offset_base, s['baseline'], width,
                               color=gen_colors[gen], alpha=0.4,
                               hatch=hatches['baseline'],
                               edgecolor='black', linewidth=0.5,
                               label=f'{gen.upper()} baseline (σ=0)')
            bar_probe = ax.bar(x + offset_probe, s['ood_probe'], width,
                                color=gen_colors[gen], alpha=1.0,
                                hatch=hatches['ood_probe'],
                                edgecolor='black', linewidth=0.5,
                                label=f'{gen.upper()} OOD probe')

            if ax is axes[0]:
                handles += [bar_base, bar_probe]
                labels += [f'{gen.upper()} baseline (σ=0)', f'{gen.upper()} OOD probe']

        ax.set_xticks(x)
        ax.set_xticklabels(label_cols, rotation=15, ha='right')
        ax.set_title(f'{model}', fontsize=12, fontweight='bold')
        ax.set_ylabel(ylabel, fontsize=11)
        ax.grid(alpha=0.3, axis='y')

    fig.legend(handles, labels, loc='lower center', ncol=4,
               fontsize=9, bbox_to_anchor=(0.5, -0.08), frameon=False)

    if filename:
        fig.savefig(filename, dpi=dpi, bbox_inches='tight')
    return fig


fig_l2_unc = plot_level2_vs_baseline('uncertainty', 'Uncertainty',
                                      filename='level2_vs_baseline_uncertainty.png')
plt.show()
plt.close(fig_l2_unc)

fig_l2_amb = plot_level2_vs_baseline('ambiguity', 'Ambiguity (%)',
                                      filename='level2_vs_baseline_ambiguity.png')
plt.show()
plt.close(fig_l2_amb)

In [ ]:
def sweep_margin_frac(generator_key, label, model='bnn',
                       margin_fracs=(0.0, 0.05, 0.10, 0.15, 0.20, 0.30, 0.50)):
    n = STRESS_CONFIG[generator_key]
    seed_base = seed_offset[generator_key] + n
    j = label_cols.index(label)
    rows = []
    for mf in margin_fracs:
        probes = build_extreme_probes(generator_key, margin_frac=mf)
        prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
            probes,
            seed_bnn=GLOBAL_SEED + 32_000 + seed_base,
            seed_mlp=GLOBAL_SEED + 42_000 + seed_base,
        )
        prob = prob_bnn if model == 'bnn' else prob_mlp
        std = std_bnn if model == 'bnn' else std_mlp
        rows.append({
            'generator': generator_key, 'label': label, 'model': model, 'margin_frac': mf,
            'mean_p': prob[:, j].mean(), 'min_p': prob[:, j].min(), 'max_p': prob[:, j].max(),
            'mean_std': std[:, j].mean(),
        })
    return pd.DataFrame(rows)


margin_sweep_records = []
for gen in ['ctgan', 'tvae']:
    for lab in ['cancer', 'stroke', 'HA', 'diabetes']:
        for model in ['bnn', 'mlp']:
            margin_sweep_records.append(sweep_margin_frac(gen, lab, model))
margin_sweep_df = pd.concat(margin_sweep_records, ignore_index=True)
margin_sweep_df


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_style("whitegrid")

outcomes = ['cancer', 'stroke', 'HA', 'diabetes']
outcome_titles = {'cancer': 'Cancer', 'stroke': 'Stroke', 'HA': 'Heart Attack', 'diabetes': 'Diabetes'}
row_ylabels = {'mean_p': 'Mean predicted probability', 'mean_std': 'Mean predictive std'}

model_markers = {'bnn': 'o', 'mlp': '^'}
model_ls = {'bnn': '-', 'mlp': '--'}
model_fill = {'bnn': True, 'mlp': False}  
model_size = {'bnn': 6, 'mlp': 8}          

fig, axes = plt.subplots(2, 4, figsize=(15, 7), sharex=True)

for col, lab in enumerate(outcomes):
    for row, ycol in enumerate(['mean_p', 'mean_std']):
        ax = axes[row, col]
        for gen in ['ctgan', 'tvae']:
            for model in ['bnn', 'mlp']:
                sub = margin_sweep_df[
                    (margin_sweep_df['label'] == lab) &
                    (margin_sweep_df['generator'] == gen) &
                    (margin_sweep_df['model'] == model)
                ].sort_values('margin_frac')

                facecolor = gen_colors[gen] if model_fill[model] else 'white'
                ax.plot(sub['margin_frac'], sub[ycol],
                        marker=model_markers[model],
                        markersize=model_size[model],
                        markerfacecolor=facecolor,
                        markeredgecolor=gen_colors[gen],
                        markeredgewidth=1.6,
                        linestyle=model_ls[model], linewidth=1.8,
                        color=gen_colors[gen], alpha=0.9,
                        label=f'{gen.upper()} – {model.upper()}')

        if row == 0:
            ax.axhline(0.5, color='gray', linewidth=1, linestyle=':', alpha=0.7)
            ax.set_title(outcome_titles[lab], fontsize=12, fontweight='bold')
        ax.grid(alpha=0.3)
        ax.spines[['top', 'right']].set_visible(False)
        if col == 0:
            ax.set_ylabel(row_ylabels[ycol], fontsize=10)
        if row == 1:
            ax.set_xlabel('margin_frac', fontsize=10)

handles, leg_labels = axes[0, 0].get_legend_handles_labels()
chance_line = plt.Line2D([0], [0], color='gray', linestyle=':', linewidth=1, label='(0.5)')
fig.legend(handles + [chance_line], leg_labels + ['(0.5)'],
           loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.06),
           frameon=False, fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 1])
plt.show()

In [ ]:
def build_single_feature_probe(generator_key, feature, n_patients=20, margin_frac=0.15):
    """Diagnostic: shifts a SINGLE feature to the extreme, the others stay unchanged."""
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    rng = np.random.default_rng(GLOBAL_SEED + 60_000)
    patient_idx = rng.choice(synth_df.index, size=min(n_patients, len(synth_df)), replace=False)
    probes = synth_df.loc[patient_idx].copy()
    train_min, train_max = REF_RANGE[feature]
    margin = margin_frac * (train_max - train_min)
    probes[feature] = train_max + margin
    return probes.reset_index(drop=True)


def single_feature_sweep(generator_key, label, model='bnn'):
    n = STRESS_CONFIG[generator_key]
    seed_base = seed_offset[generator_key] + n
    j = label_cols.index(label)
    rows = []
    for feat in CONTINUOUS_FEATURES:
        probes = build_single_feature_probe(generator_key, feat)
        prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
            probes,
            seed_bnn=GLOBAL_SEED + 32_000 + seed_base,
            seed_mlp=GLOBAL_SEED + 42_000 + seed_base,
        )
        prob = prob_bnn if model == 'bnn' else prob_mlp
        std = std_bnn if model == 'bnn' else std_mlp
        rows.append({
            'generator': generator_key, 'label': label, 'model': model, 'feature': feat,
            'mean_p': prob[:, j].mean(), 'mean_std': std[:, j].mean(),
        })
    return pd.DataFrame(rows)


single_feature_records = []
for gen in ['ctgan', 'tvae']:
    for lab in ['cancer', 'stroke', 'HA','diabetes']:
        for model in ['bnn', 'mlp']:
            single_feature_records.append(single_feature_sweep(gen, lab, model))
single_feature_df = pd.concat(single_feature_records, ignore_index=True)
single_feature_df.sort_values(['label', 'model', 'generator', 'mean_p'])


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set_style("whitegrid")

labels = ['cancer', 'stroke', 'HA', 'diabetes']
label_titles = {'cancer': 'Cancer', 'stroke': 'Stroke', 'HA': 'Heart Attack', 'diabetes': 'Diabetes'}
models = ['bnn', 'mlp']
model_titles = {'bnn': 'BNN', 'mlp': 'MLP Dropout'}
gen_colors = {'ctgan': '#1f4e79', 'tvae': '#e07b39'}


def plot_single_feature_group(label_group, filename=None):
    fig, axes = plt.subplots(len(label_group), 2, figsize=(13, 7 * len(label_group) / 2), sharex=False)
    if len(label_group) == 1:
        axes = axes.reshape(1, 2)

    for row, lab in enumerate(label_group):
        row_data = single_feature_df[single_feature_df['label'] == lab]
        y_max = min(1.0, row_data['mean_p'].max() * 1.12)

        for col, model in enumerate(models):
            ax = axes[row, col]
            sub = single_feature_df[(single_feature_df['label'] == lab) & (single_feature_df['model'] == model)]
            order = sub[sub['generator'] == 'ctgan'].sort_values('mean_p')['feature']
            x = np.arange(len(order))
            width = 0.35

            for i, gen in enumerate(['ctgan', 'tvae']):
                s = sub[sub['generator'] == gen].set_index('feature').reindex(order)
                ax.bar(x + (i - 0.5) * width, s['mean_p'], width,
                       color=gen_colors[gen], label=gen.upper(), edgecolor='white', linewidth=0.5)

            ax.axhline(0.5, color='gray', linewidth=1, linestyle='--', alpha=0.7)
            ax.set_xticks(x)
            ax.set_xticklabels(order, rotation=45, ha='right', fontsize=8)
            ax.set_title(model_titles[model], fontsize=11, fontweight='bold')
            ax.set_ylabel('Mean predicted probability' if col == 0 else '')
            ax.set_ylim(0, y_max)
            ax.grid(alpha=0.3, axis='y')
            ax.spines[['top', 'right']].set_visible(False)

        axes[row, 0].annotate(label_titles[lab], xy=(-0.28, 0.5), xycoords='axes fraction',
                               fontsize=13, fontweight='bold', rotation=90,
                               ha='center', va='center')

    handles, leg_labels = axes[0, 0].get_legend_handles_labels()
    chance_line = plt.Line2D([0], [0], color='gray', linestyle='--', linewidth=1, label='(0.5)')
    fig.legend(handles + [chance_line], leg_labels + ['(0.5)'],
               loc='upper center', ncol=3, bbox_to_anchor=(0.5, 1.02), frameon=False, fontsize=10)

    plt.tight_layout(rect=[0.02, 0, 1, 1])
    if filename:
        plt.savefig(filename, bbox_inches='tight')
    plt.show()


plot_single_feature_group(['cancer', 'stroke'], filename='mean_predicted_probability_per_feature_part1.pdf')

plot_single_feature_group(['HA', 'diabetes'], filename='mean_predicted_probability_per_feature_part2.pdf')

In [ ]:
def plot_level2_vs_baseline(metric, ylabel, filename=None):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, model in zip(axes, ['BNN', 'MLP']):
        sub = level2_vs_baseline[
            (level2_vs_baseline['model'] == model) &
            (level2_vs_baseline['metric'] == metric)
        ]
        x = np.arange(len(label_cols))
        width = 0.2
        for i, gen in enumerate(['ctgan', 'tvae']):
            s = sub[sub['generator'] == gen].set_index('label').reindex(label_cols)
            ax.bar(x + (i * 2) * width - width * 1.5, s['baseline'], width,
                   color=gen_colors[gen], alpha=0.4, label=f'{gen.upper()} baseline (sigma=0)')
            ax.bar(x + (i * 2 + 1) * width - width * 1.5, s['ood_probe'], width,
                   color=gen_colors[gen], alpha=1.0, label=f'{gen.upper()} OOD probe')
        ax.set_xticks(x)
        ax.set_xticklabels(label_cols)
        ax.set_title(f'{model} — baseline vs OOD probe (Level 2)')
        ax.set_ylabel(ylabel)
        ax.legend(fontsize=7)
        ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
    return fig


fig_l2_unc = plot_level2_vs_baseline('uncertainty', 'uncertainty',
                                     filename='level2_vs_baseline_uncertainty.png')
plt.show()
fig_l2_amb = plot_level2_vs_baseline('ambiguity', 'ambiguity (%)',
                                     filename='level2_vs_baseline_ambiguity.png')
plt.show()

In [ ]:
ha_mlp_focus = (
    single_feature_df[(single_feature_df['label'] == 'HA') & (single_feature_df['model'] == 'mlp')]
    .sort_values(['generator', 'mean_p'])
)
print("Feature che, da sole, spingono mean_p sotto 0.5 per HA-MLP:")
print(ha_mlp_focus[ha_mlp_focus['mean_p'] < 0.5][['generator', 'feature', 'mean_p', 'mean_std']])
ha_mlp_focus


In [ ]:
ha_bnn_focus = (
    single_feature_df[(single_feature_df['label'] == 'HA') & (single_feature_df['model'] == 'bnn')]
    .sort_values(['generator', 'mean_p'])
)
print("Feature che, da sole, spingono mean_p sotto 0.5 per HA-bnn:")
print(ha_mlp_focus[ha_mlp_focus['mean_p'] < 0.5][['generator', 'feature', 'mean_p', 'mean_std']])
ha_bnn_focus

## 7. LEVEL 3 — FEATURE PERMUTATION 

In [23]:
def permute_feature(df, feature, seed_offset_extra=0):
    rng = np.random.default_rng(GLOBAL_SEED + 70_000 + seed_offset_extra)
    df_perm = df.copy()
    df_perm[feature] = rng.permutation(df_perm[feature].values)
    return df_perm


def run_level3(generator_key, feature, n_repeats=10, patient_ids_detail=None):
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    seed_base = seed_offset[generator_key] + n
    records, patient_records = [], []
    for r in range(n_repeats):
        df_perm = permute_feature(synth_df, feature, seed_offset_extra=r)
        prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
            df_perm,
            seed_bnn=GLOBAL_SEED + 33_000 + seed_base + r,
            seed_mlp=GLOBAL_SEED + 43_000 + seed_base + r,
        )
        summary = summarize_stress_level(prob_bnn, std_bnn, prob_mlp, std_mlp, label_cols)
        summary['repeat'] = r
        summary['generator'] = generator_key
        summary['feature_permuted'] = feature
        summary['label'] = summary.index
        records.append(summary.reset_index(drop=True))

        if patient_ids_detail:
            p_summary = summarize_stress_level_patients(prob_bnn, std_bnn, prob_mlp, std_mlp,
                                                        label_cols, patient_ids_detail)
            p_summary['repeat'] = r
            p_summary['generator'] = generator_key
            p_summary['feature_permuted'] = feature
            patient_records.append(p_summary)

    patient_df = pd.concat(patient_records, ignore_index=True) if patient_records else None
    return pd.concat(records, ignore_index=True), patient_df


In [ ]:
level3_records = []
level3_patient_records = []

for gen in ['ctgan', 'tvae']:
    for f in CONTINUOUS_FEATURES:
        agg, pat = run_level3(gen, f, patient_ids_detail=CASE_STUDY_PATIENT_IDS)
        level3_records.append(agg)
        if pat is not None:
            level3_patient_records.append(pat)

level3_all = pd.concat(level3_records, ignore_index=True)
level3_patients_raw = pd.concat(level3_patient_records, ignore_index=True)

level3_chol = level3_all[level3_all['feature_permuted'] == 'chol']

level3_summary = (level3_all.groupby(['generator', 'feature_permuted', 'label'])
                   [['uncertainty_bnn', 'ambiguity_bnn',
                     'uncertainty_mlp', 'ambiguity_mlp', 'disagreement']]
                   .agg(['mean', 'std']))

level3_patients = (
    level3_patients_raw
    .groupby(['patient_id', 'label', 'generator'])
    [['uncertainty_bnn', 'ambiguity_bnn', 'uncertainty_mlp', 'ambiguity_mlp']]
    .mean()
    .reset_index()
)

level3_summary.head(10)

In [ ]:
baseline_by_gen = (
    baseline_l1
    .groupby('generator')[['uncertainty_bnn', 'ambiguity_bnn', 'uncertainty_mlp', 'ambiguity_mlp']]
    .mean()
)

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.titleweight': 'bold',
    'axes.edgecolor': '#444444',
    'axes.linewidth': 0.8,
})

fig, axes = plt.subplots(1, 2, figsize=(13, 7), sharex=False)

for ax, model_col, title in zip(axes, ['uncertainty_bnn', 'uncertainty_mlp'], ['BNN', 'MLP Dropout']):
    means = (level3_all.groupby(['generator', 'feature_permuted'])[model_col]
             .mean().unstack('generator'))
    stds = (level3_all.groupby(['generator', 'feature_permuted', 'repeat'])[model_col].mean()
            .groupby(['generator', 'feature_permuted']).std()
            .unstack('generator'))

    delta = means.sub(baseline_by_gen[model_col], axis=1)
    order = delta.mean(axis=1).sort_values(ascending=True).index
    delta = delta.reindex(order)
    stds = stds.reindex(order)

    y = np.arange(len(order))
    offset = 0.16

    for yi in y:
        if yi % 2 == 0:
            ax.axhspan(yi - 0.5, yi + 0.5, color='#f2f2f2', zorder=0)

    x_min = (delta.values - stds.values).min()
    x_max = (delta.values + stds.values).max()
    x_range = x_max - x_min
    right_edge_thresh = x_max - 0.15 * x_range  

    for i, gen in enumerate(['ctgan', 'tvae']):
        yy = y + (i - 0.5) * offset * 2
        lo, hi = delta[gen] - stds[gen], delta[gen] + stds[gen]
        significant = (lo > 0) | (hi < 0)
        sig_mask = significant.values

        ax.errorbar(delta[gen], yy, xerr=stds[gen], fmt='none',
                    ecolor=gen_colors[gen], elinewidth=1.3, capsize=3, zorder=2, alpha=0.9)

        ax.scatter(delta[gen][sig_mask], yy[sig_mask], color=gen_colors[gen],
                   s=55, zorder=3, edgecolor='black', linewidth=0.6,
                   label=f'{gen.upper()} (significant effect)')
        ax.scatter(delta[gen][~sig_mask], yy[~sig_mask], facecolor='white',
                   edgecolor=gen_colors[gen], linewidth=1.3, s=55, zorder=3, alpha=0.85,
                   label=f'{gen.upper()} (not significant)')

        for xv, xe, yv in zip(delta[gen], stds[gen], yy):
            if xv > right_edge_thresh:
                ax.annotate(f'{xv:+.3f}', (xv - xe, yv), xytext=(-6, 0), textcoords='offset points',
                           ha='right', va='center', fontsize=7, color=gen_colors[gen],
                           bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.75))
            else:
                ax.annotate(f'{xv:+.3f}', (xv + xe, yv), xytext=(6, 0), textcoords='offset points',
                           ha='left', va='center', fontsize=7, color=gen_colors[gen],
                           bbox=dict(boxstyle='round,pad=0.15', facecolor='white', edgecolor='none', alpha=0.75))

    ax.axvline(0, color='#222222', linewidth=1)
    ax.set_yticks(y)
    ax.set_yticklabels(order, fontsize=9.5)
    ax.set_ylim(-0.7, len(order) - 0.3)
    ax.set_title(title, pad=10)
    ax.set_xlabel('Δ uncertainty')
    ax.grid(alpha=0.25, axis='x', linewidth=0.6)
    ax.margins(x=0.2)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

handles, labels_legend = axes[0].get_legend_handles_labels()
handles2, labels2 = axes[1].get_legend_handles_labels()
for h, l in zip(handles2, labels2):
    if l not in labels_legend:
        handles.append(h); labels_legend.append(l)

seen = set(); h2, l2 = [], []
for h, l in zip(handles, labels_legend):
    if l not in seen:
        h2.append(h); l2.append(l); seen.add(l)
order_map = {'CTGAN (significant)': 0, 'TVAE (significant)': 1,
             'CTGAN (not significant)': 2, 'TVAE (not significant)': 3}
sorted_pairs = sorted(zip(h2, l2), key=lambda p: order_map.get(p[1], 99))
h2, l2 = zip(*sorted_pairs)

fig.legend(h2, l2, loc='upper center', ncol=4, frameon=False,
           bbox_to_anchor=(0.5, 0.98), fontsize=8.5)

plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()

## 8. METRICS


In [ ]:
def _trapz_integral(y, x):
    if hasattr(np, "trapezoid"):
        return np.trapezoid(y, x)
    return np.trapz(y, x)  


def stress_auc(df_level1, generator_key, label, model_col_prefix, metric='uncertainty'):
    subset = df_level1[(df_level1['generator'] == generator_key) & (df_level1['label'] == label)]
    subset = subset.sort_values('sigma_level')
    x = subset['sigma_level'].values
    y = subset[f'{metric}_{model_col_prefix}'].values
    if len(x) < 2 or x.max() == x.min():
        return np.nan
    return _trapz_integral(y, x) / (x.max() - x.min())


def stress_sensitivity(df_level1, generator_key, label, model_col_prefix, metric='uncertainty'):
    """Slope and Pearson correlation of metric ~ sigma_level."""
    subset = df_level1[(df_level1['generator'] == generator_key) & (df_level1['label'] == label)]
    subset = subset.sort_values('sigma_level')
    x = subset['sigma_level'].values
    y = subset[f'{metric}_{model_col_prefix}'].values
    if len(x) < 2:
        return np.nan, np.nan
    slope, _ = np.polyfit(x, y, 1)
    corr = np.corrcoef(x, y)[0, 1] if np.std(y) > 0 and np.std(x) > 0 else np.nan
    return float(slope), float(corr)


auc_records = []
for gen in ['ctgan', 'tvae']:
    for lab in label_cols:
        row = {'generator': gen, 'label': lab}
        for metric in ['uncertainty', 'ambiguity']:
            for model in ['bnn', 'mlp']:
                slope, corr = stress_sensitivity(level1_aggregate, gen, lab, model, metric)
                row[f'auc_{metric}_{model}'] = stress_auc(level1_aggregate, gen, lab, model, metric)
                row[f'slope_{metric}_{model}'] = slope
                row[f'corr_{metric}_{model}'] = corr
        auc_records.append(row)

stress_auc_df = pd.DataFrame(auc_records)
stress_auc_df['auc_gap_uncertainty'] = stress_auc_df['auc_uncertainty_mlp'] - stress_auc_df['auc_uncertainty_bnn']
stress_auc_df['auc_gap_ambiguity'] = stress_auc_df['auc_ambiguity_mlp'] - stress_auc_df['auc_ambiguity_bnn']
stress_auc_df['slope_gap_uncertainty'] = stress_auc_df['slope_uncertainty_mlp'] - stress_auc_df['slope_uncertainty_bnn']
stress_auc_df['slope_gap_ambiguity'] = stress_auc_df['slope_ambiguity_mlp'] - stress_auc_df['slope_ambiguity_bnn']
stress_auc_df


## 9. Patient Level

### 9.1 LEVEL 1

In [ ]:
def run_level1_patients(generator_key, patient_ids, sigma_levels=None):
    sigma_levels = sigma_levels if sigma_levels is not None else SIGMA_LEVELS
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    seed_base = seed_offset[generator_key] + n
    records = []
    for k, sigma in enumerate(sigma_levels):

        df_pert = inject_gaussian_noise(synth_df, CONTINUOUS_FEATURES, sigma, seed_offset_extra=k)
        prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
            df_pert,
            seed_bnn=GLOBAL_SEED + 30_000 + seed_base + k,
            seed_mlp=GLOBAL_SEED + 40_000 + seed_base + k,
        )
        summary = summarize_stress_level_patients(prob_bnn, std_bnn, prob_mlp, std_mlp,
                                                  label_cols, patient_ids)
        summary['sigma_level'] = sigma
        summary['generator'] = generator_key
        records.append(summary)
    return pd.concat(records, ignore_index=True)


level1_patients = pd.concat(
    [run_level1_patients('ctgan', CASE_STUDY_PATIENT_IDS),
     run_level1_patients('tvae', CASE_STUDY_PATIENT_IDS)],
    ignore_index=True,
)

level1_patients.head()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

def plot_patient_trajectories(generator_key, metric_prefix, ylabel, filename=None):
    """
    Plot uncertainty/ambiguity trajectories across sigma_level for a case-study
    patient against the full cohort, split by clinical label (rows) and model (columns).
    """
    n_labels = len(label_cols)
    fig, axes = plt.subplots(
        n_labels, 2, figsize=(11, 2.9 * n_labels), sharex=True
    )
    if n_labels == 1:
        axes = axes.reshape(1, 2)

    ref_color = '#C1121F'     
    cohort_color = gen_colors[generator_key]  
    baseline_style = dict(color='#333333', linestyle=':', linewidth=0.9, alpha=0.6)

    for row, label in enumerate(label_cols):
        for col_idx, model in enumerate(['bnn', 'mlp']):
            ax = axes[row, col_idx]
            sub_all = level1_patients[
                (level1_patients['generator'] == generator_key) &
                (level1_patients['label'] == label)
            ]

            for pid in CASE_STUDY_PATIENT_IDS:
                if pid == REFERENCE_PATIENT_ID:
                    continue
                sub = sub_all[sub_all['patient_id'] == pid].sort_values('sigma_level')
                if sub.empty:
                    continue
                ax.plot(sub['sigma_level'], sub[f'{metric_prefix}_{model}'],
                        color=cohort_color, alpha=0.4, linewidth=0.9, zorder=1)

            ref_sub = sub_all[sub_all['patient_id'] == REFERENCE_PATIENT_ID].sort_values('sigma_level')
            if not ref_sub.empty:
                ax.plot(ref_sub['sigma_level'], ref_sub[f'{metric_prefix}_{model}'],
                        color=ref_color, linewidth=2.2, marker='o', markersize=4,
                        zorder=3)
                baseline_val = ref_sub[f'{metric_prefix}_{model}'].iloc[0]
                ax.axhline(baseline_val, **baseline_style, zorder=2)

            if row == 0:
                ax.set_title('BNN' if model == 'bnn' else 'MLP Dropout',
                              fontsize=12, fontweight='bold', pad=10)
            if row == n_labels - 1:
                ax.set_xlabel('sigma_level')
            if col_idx == 0:
                ax.set_ylabel(ylabel)
                ax.text(-0.28, 0.5, label, transform=ax.transAxes,
                        fontsize=11, fontweight='bold', rotation=90,
                        va='center', ha='center')
            ax.grid(alpha=0.25)

    legend_handles = [
        mlines.Line2D([], [], color=ref_color, linewidth=2.2, marker='o',
                      markersize=4, label=f'Patient {REFERENCE_PATIENT_ID}'),
        mlines.Line2D([], [], color=gen_colors[generator_key], linewidth=1.2, alpha=0.5,
                      label='Other patients'),
    ]
    fig.legend(handles=legend_handles, loc='upper center', ncol=3, frameon=False,
               bbox_to_anchor=(0.5, 1.03), fontsize=10)

    fig.tight_layout(rect=[0.02, 0, 1, 0.97])
    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    return fig


for gen in ['ctgan', 'tvae']:
    fig_traj_unc = plot_patient_trajectories(
        gen, 'uncertainty', 'uncertainty',
        filename=f'level1_patient_trajectories_{gen}_uncertainty.png'
    )
    plt.show()
    fig_traj_amb = plot_patient_trajectories(
        gen, 'ambiguity', 'ambiguity (%)',
        filename=f'level1_patient_trajectories_{gen}_ambiguity.png'
    )
    plt.show()

### 9.2 LEVEL 2 


In [ ]:
def build_extreme_probes_patients(generator_key, patient_ids, features=None, margin_frac=0.15):
    n = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n] if generator_key == 'ctgan' else synthetic_tvae[n]
    features = features or CONTINUOUS_FEATURES
    probe_df = synth_df.copy()
    for pid in patient_ids:
        if pid not in probe_df.index:
            continue
        for f in features:
            train_min, train_max = REF_RANGE[f]
            margin = margin_frac * (train_max - train_min)
            probe_df.loc[pid, f] = train_max + margin
    return probe_df


def run_level2_patients(generator_key, patient_ids, margin_frac=0.15):
    n = STRESS_CONFIG[generator_key]
    seed_base = seed_offset[generator_key] + n
    df_probe = build_extreme_probes_patients(generator_key, patient_ids, margin_frac=margin_frac)
    prob_bnn, std_bnn, prob_mlp, std_mlp = predict_on_dataframe(
        df_probe,
        seed_bnn=GLOBAL_SEED + 32_000 + seed_base,
        seed_mlp=GLOBAL_SEED + 42_000 + seed_base,
    )
    summary = summarize_stress_level_patients(prob_bnn, std_bnn, prob_mlp, std_mlp,
                                              label_cols, patient_ids)
    summary['generator'] = generator_key
    return summary


level2_patients = pd.concat(
    [run_level2_patients('ctgan', CASE_STUDY_PATIENT_IDS),
     run_level2_patients('tvae', CASE_STUDY_PATIENT_IDS)],
    ignore_index=True,
)

level2_patients.head()


### 9.3 LEVEL 3 

In [ ]:
level3_patients.head()

,patient_id,label,generator,uncertainty_bnn,ambiguity_bnn,uncertainty_mlp,ambiguity_mlp
0,28,HA,ctgan,1.175384,0.962570,4.795351,0.955630
1,28,HA,tvae,0.440647,0.641151,5.440815,0.683971
2,28,cancer,ctgan,2.088890,0.919062,8.845075,0.996236
3,28,cancer,tvae,0.740090,0.434011,4.541100,0.647560
4,28,diabetes,ctgan,1.450436,0.940183,6.346428,0.977751


In [ ]:
def patient_delta_table(patient_ids):
    rows = []
    for pid in patient_ids:
        for gen in ['ctgan', 'tvae']:
            for lab in label_cols:
                for model in ['bnn', 'mlp']:
                    for metric in ['uncertainty', 'ambiguity']:
                        col = f'{metric}_{model}'
                        base = level1_patients[
                            (level1_patients['patient_id'] == pid) &
                            (level1_patients['generator'] == gen) &
                            (level1_patients['label'] == lab) &
                            (level1_patients['sigma_level'] == 0.0)
                        ][col].mean()
                        l1_max = level1_patients[
                            (level1_patients['patient_id'] == pid) &
                            (level1_patients['generator'] == gen) &
                            (level1_patients['label'] == lab) &
                            (level1_patients['sigma_level'] == 1.0)
                        ][col].mean()
                        l2_val = level2_patients[
                            (level2_patients['patient_id'] == pid) &
                            (level2_patients['generator'] == gen) &
                            (level2_patients['label'] == lab)
                        ][col].mean()
                        l3_val = level3_patients[
                            (level3_patients['patient_id'] == pid) &
                            (level3_patients['generator'] == gen) &
                            (level3_patients['label'] == lab)
                        ][col].mean()
                        rows.append({
                            'patient_id': pid,
                            'label': lab,
                            'generator': gen,
                            'model': model.upper(),
                            'metric': metric,
                            'baseline': base,
                            'delta_L1_sigma_1': l1_max - base,
                            'delta_L2_all_features': l2_val - base,
                            'delta_L3_permutation': l3_val - base,
                            'is_reference': pid == REFERENCE_PATIENT_ID,
                        })
    return pd.DataFrame(rows)


patient_delta_df = patient_delta_table(CASE_STUDY_PATIENT_IDS)
patient_delta_df.sort_values(['generator', 'model', 'metric', 'label', 'patient_id'])


In [ ]:
import matplotlib.pyplot as plt

def plot_level2_delta_uncertainty_vs_ambiguity(patient_delta_df, filename=None):
    """
    Patient-level scatter for Level 2 analysis:
    x = delta uncertainty (OOD probe - baseline)
    y = delta ambiguity (OOD probe - baseline)

    One point per patient, averaged across labels.
    Panels split by generator (rows) and model (columns).
    """
    plot_df = (
        patient_delta_df
        .pivot_table(
            index=['patient_id', 'generator', 'model', 'is_reference'],
            columns='metric',
            values='delta_L2_all_features',
            aggfunc='mean'
        )
        .reset_index()
    )

    ref_color = '#C1121F'  
    generators = ['ctgan', 'tvae']
    models = ['BNN', 'MLP']

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.5), sharex='row', sharey='row')

    for r, gen in enumerate(generators):
        for c, model in enumerate(models):
            ax = axes[r, c]
            sub = plot_df[
                (plot_df['generator'] == gen) &
                (plot_df['model'] == model)
            ]

            ax.axhline(0, color='black', linewidth=0.8, alpha=0.7, zorder=1)
            ax.axvline(0, color='black', linewidth=0.8, alpha=0.7, zorder=1)

            normal = sub[~sub['is_reference']]
            ref = sub[sub['is_reference']]

            ax.scatter(
                normal['uncertainty'], normal['ambiguity'],
                s=65, color=gen_colors[gen], alpha=0.7,
                edgecolor='white', linewidth=0.6,
                label='Other patients', zorder=2
            )

            if not ref.empty:
                ax.scatter(
                    ref['uncertainty'], ref['ambiguity'],
                    s=70, color=ref_color, edgecolor='black', linewidth=0.7,
                    label=f'Patient {REFERENCE_PATIENT_ID}', zorder=3
                )


            if not ref.empty:
                for _, row in ref.iterrows():
                    ax.annotate(
                        f"{int(row['patient_id'])}",
                        (row['uncertainty'], row['ambiguity']),
                        xytext=(4, 4),
                        textcoords='offset points',
                        fontsize=7,
                        fontweight='bold',
                        color=ref_color,
                        zorder=4
                    )

            ax.set_title('BNN' if model == 'BNN' else 'MLP Dropout',
                         fontsize=12, fontweight='bold')
            ax.grid(alpha=0.2, zorder=0)

            if r == 1:
                ax.set_xlabel('Δ uncertainty')
            if c == 0:
                ax.set_ylabel('Δ ambiguity')
                ax.text(-0.24, 0.5, gen.upper(), transform=ax.transAxes,
                        fontsize=11, fontweight='bold', rotation=90,
                        va='center', ha='center')

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', ncol=2, frameon=False,
               bbox_to_anchor=(0.5, 0.975), fontsize=10)

    fig.tight_layout(rect=[0.02, 0, 1, 0.94])

    if filename:
        plt.savefig(filename, dpi=300, bbox_inches='tight')

    return fig


fig_l2_scatter = plot_level2_delta_uncertainty_vs_ambiguity(
    patient_delta_df,
    filename='level2_delta_uncertainty_vs_ambiguity_patient.png'
)
plt.show()

In [ ]:
def plot_level3_patient_feature_heatmap(level3_patients_raw, level1_patients, metric='uncertainty',
                                          patient_order=None, filename=None, shared_vmax=None):
    """Level 3 — patient x permuted-feature heatmap: delta of `metric`
    (mean over BNN/MLP, generator, label, and repeat) relative to baseline.

    shared_vmax: if provided, use this fixed vmax instead of computing it
    per-plot, so uncertainty and ambiguity heatmaps become directly comparable
    (at the cost of ambiguity looking much fainter, since its deltas are smaller)."""
    patient_order = patient_order or CASE_STUDY_PATIENT_IDS
    col_bnn, col_mlp = f'{metric}_bnn', f'{metric}_mlp'

    baseline = level1_patients[level1_patients['sigma_level'] == 0.0].copy()
    baseline['val_mean'] = baseline[[col_bnn, col_mlp]].mean(axis=1)
    baseline = baseline[['patient_id', 'generator', 'label', 'val_mean']]

    l3 = level3_patients_raw.copy()
    l3['val_mean'] = l3[[col_bnn, col_mlp]].mean(axis=1)

    merged = l3.merge(baseline, on=['patient_id', 'generator', 'label'], suffixes=('', '_base'))
    merged['delta'] = merged['val_mean'] - merged['val_mean_base']

    heat = (merged.groupby(['patient_id', 'feature_permuted'])['delta']
            .mean().reset_index()
            .pivot(index='patient_id', columns='feature_permuted', values='delta')
            .reindex(patient_order))

    fig, ax = plt.subplots(figsize=(1.0 * len(heat.columns) + 3, 0.5 * len(heat.index) + 2))
    vmax = shared_vmax if shared_vmax is not None else np.nanmax(np.abs(heat.values))
    im = ax.imshow(heat.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')
    ax.grid(False)  

    n_rows, n_cols = heat.shape
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(heat.columns, rotation=45, ha='right')
    ax.set_yticks(np.arange(n_rows))
    tick_labels = [f"{pid}{' *' if pid == REFERENCE_PATIENT_ID else ''}" for pid in heat.index]
    ax.set_yticklabels(tick_labels)

  
    ref_idx = list(heat.index).index(REFERENCE_PATIENT_ID)
    ax.add_patch(plt.Rectangle((-0.5, ref_idx - 0.5), n_cols, 1,
                                fill=False, edgecolor='black', linewidth=2))

    ax.set_xlabel('Permuted feature')
    ax.set_ylabel('patient_id (* = reference patient)')
    fig.colorbar(im, ax=ax, label=f'Δ {metric} (mean over BNN/MLP, generators, repeats)')
    fig.tight_layout()
    if filename:
        plt.savefig(filename, dpi=150, bbox_inches='tight')
    return fig


fig_l3 = plot_level3_patient_feature_heatmap(level3_patients_raw, level1_patients, metric='uncertainty',
                                              filename='level3_patient_feature_heatmap_uncertainty.png')
plt.show()

fig_l3_amb = plot_level3_patient_feature_heatmap(level3_patients_raw, level1_patients, metric='ambiguity',
                                                  filename='level3_patient_feature_heatmap_ambiguity.png')
plt.show()

### 9.5 ROBUSTNESS

In [ ]:
def patient_slope(df_patients, pid, gen, label, model, metric='uncertainty'):
    """Slope of metric ~ sigma for ONE patient, same principle as stress_sensitivity()
    but applied row by row instead of on the aggregate."""
    sub = df_patients[
        (df_patients['patient_id'] == pid) &
        (df_patients['generator'] == gen) &
        (df_patients['label'] == label)
    ].sort_values('sigma_level')
    x = sub['sigma_level'].values
    y = sub[f'{metric}_{model}'].values
    if len(x) < 2 or np.std(y) == 0:
        return np.nan
    slope, _ = np.polyfit(x, y, 1)
    return float(slope)


def build_patient_slope_table(df_patients, patient_ids, generators=('ctgan', 'tvae'),
                               models=('bnn', 'mlp'), metrics=('uncertainty', 'ambiguity')):
    records = []
    for pid in patient_ids:
        for gen in generators:
            for lab in label_cols:
                for model in models:
                    for metric in metrics:
                        records.append({
                            'patient_id': pid, 'generator': gen, 'label': lab,
                            'model': model, 'metric': metric,
                            'slope': patient_slope(df_patients, pid, gen, lab, model, metric),
                        })
    return pd.DataFrame(records)


patient_slope_table = build_patient_slope_table(level1_patients, CASE_STUDY_PATIENT_IDS)
patient_slope_table.head()


from scipy.stats import binomtest

def concordance_table(patient_slope_table, stress_auc_df):
    """For each label/generator/model/metric: % of patients whose sign agrees
    with the aggregate slope, and p-value of a sign test against H0: random sign (p=0.5)."""
    records = []
    for gen in ['ctgan', 'tvae']:
        for lab in label_cols:
            for model in ['bnn', 'mlp']:
                for metric in ['uncertainty', 'ambiguity']:
                    agg_row = stress_auc_df[
                        (stress_auc_df['generator'] == gen) & (stress_auc_df['label'] == lab)
                    ]
                    agg_slope = agg_row[f'slope_{metric}_{model}'].values[0]
                    agg_sign = np.sign(agg_slope)

                    sub = patient_slope_table[
                        (patient_slope_table['generator'] == gen) &
                        (patient_slope_table['label'] == lab) &
                        (patient_slope_table['model'] == model) &
                        (patient_slope_table['metric'] == metric)
                    ].dropna(subset=['slope'])

                    n = len(sub)
                    if n == 0:
                        continue
                    n_concordant = int((np.sign(sub['slope']) == agg_sign).sum())
                    test = binomtest(n_concordant, n, 0.5, alternative='greater')

                    records.append({
                        'generator': gen, 'label': lab, 'model': model, 'metric': metric,
                        'aggregate_slope': agg_slope, 'aggregate_sign': agg_sign,
                        'n_patients': n, 'n_concordant': n_concordant,
                        'concordance_rate': n_concordant / n,
                        'sign_test_pvalue': test.pvalue,
                    })
    return pd.DataFrame(records)


concordance_df = concordance_table(patient_slope_table, stress_auc_df)
concordance_df.sort_values(['model', 'label', 'generator'])


def patient_direction_level2(level2_patients, level1_patients, patient_ids,
                              generators=('ctgan', 'tvae'), models=('bnn', 'mlp'),
                              metrics=('uncertainty', 'ambiguity')):
    """For each patient: baseline (sigma=0) vs OOD probe, sign of the delta.
    Counts how many patients follow the aggregate Level 2 pattern
    (MLP always increases; BNN decreases on 3/4 labels)."""
    records = []
    for pid in patient_ids:
        for gen in generators:
            for lab in label_cols:
                for model in models:
                    for metric in metrics:
                        col = f'{metric}_{model}'
                        base = level1_patients[
                            (level1_patients['patient_id'] == pid) &
                            (level1_patients['generator'] == gen) &
                            (level1_patients['label'] == lab) &
                            (level1_patients['sigma_level'] == 0.0)
                        ][col].mean()
                        probe = level2_patients[
                            (level2_patients['patient_id'] == pid) &
                            (level2_patients['generator'] == gen) &
                            (level2_patients['label'] == lab)
                        ][col].mean()
                        if pd.isna(base) or pd.isna(probe):
                            continue
                        records.append({
                            'patient_id': pid, 'generator': gen, 'label': lab,
                            'model': model, 'metric': metric,
                            'delta': probe - base, 'direction_correct': (probe - base) > 0,
                        })
    return pd.DataFrame(records)


level2_patient_direction = patient_direction_level2(level2_patients, level1_patients, CASE_STUDY_PATIENT_IDS)

level2_direction_summary = (
    level2_patient_direction
    .groupby(['generator', 'label', 'model', 'metric'])['direction_correct']
    .agg(n_correct='sum', n_total='count')
    .assign(pct_correct=lambda d: d['n_correct'] / d['n_total'])
    .reset_index()
)
level2_direction_summary


def patient_direction_level3(level3_patients_raw, level1_patients, patient_ids, feature='age',
                              generators=('ctgan', 'tvae'), models=('bnn', 'mlp'),
                              metrics=('uncertainty', 'ambiguity')):
    """Same principle as Level 2, applied to Level 3: compares baseline vs
    permuted for a single feature (here: age, which dominates the effect on the
    MLP Dropout in Level 3), averaging over the permutation repeats.
    
    Uses level3_patients_raw (not level3_patients) because the latter is already
    averaged over ALL permuted features together and has lost 'feature_permuted'.
    """
    baseline = level1_patients[level1_patients['sigma_level'] == 0.0]

    perm_feature = level3_patients_raw[level3_patients_raw['feature_permuted'] == feature]
    perm_mean = (
        perm_feature
        .groupby(['patient_id', 'label', 'generator'])
        [['uncertainty_bnn', 'ambiguity_bnn', 'uncertainty_mlp', 'ambiguity_mlp']]
        .mean()
        .reset_index()
    )

    records = []
    for pid in patient_ids:
        for gen in generators:
            for lab in label_cols:
                for model in models:
                    for metric in metrics:
                        col = f'{metric}_{model}'
                        base = baseline[
                            (baseline['patient_id'] == pid) &
                            (baseline['generator'] == gen) &
                            (baseline['label'] == lab)
                        ][col].mean()
                        permd = perm_mean[
                            (perm_mean['patient_id'] == pid) &
                            (perm_mean['generator'] == gen) &
                            (perm_mean['label'] == lab)
                        ][col].mean()
                        if pd.isna(base) or pd.isna(permd):
                            continue
                        records.append({
                            'patient_id': pid, 'generator': gen, 'label': lab,
                            'model': model, 'metric': metric,
                            'delta': permd - base, 'direction_correct': (permd - base) > 0,
                        })
    return pd.DataFrame(records)


level3_patient_direction = patient_direction_level3(
    level3_patients_raw, level1_patients, CASE_STUDY_PATIENT_IDS, feature='age'
)

level3_direction_summary = (
    level3_patient_direction
    .groupby(['generator', 'label', 'model', 'metric'])['direction_correct']
    .agg(n_correct='sum', n_total='count')
    .assign(pct_correct=lambda d: d['n_correct'] / d['n_total'])
    .reset_index()
)
level3_direction_summary

RANDOM_SEED_CONTROL = GLOBAL_SEED + 90_000

def sample_random_control_patients(generator_key, n=10, exclude_ids=None):
    n_synth = STRESS_CONFIG[generator_key]
    synth_df = synthetic_ctgan[n_synth] if generator_key == 'ctgan' else synthetic_tvae[n_synth]
    rng = np.random.default_rng(RANDOM_SEED_CONTROL)
    pool = synth_df.index.difference(exclude_ids or [])
    return rng.choice(pool, size=min(n, len(pool)), replace=False).tolist()

CONTROL_PATIENT_IDS_CTGAN = sample_random_control_patients('ctgan', 10, exclude_ids=CASE_STUDY_PATIENT_IDS)
CONTROL_PATIENT_IDS_TVAE  = sample_random_control_patients('tvae', 10, exclude_ids=CASE_STUDY_PATIENT_IDS)

level1_control = pd.concat([
    run_level1_patients('ctgan', CONTROL_PATIENT_IDS_CTGAN),
    run_level1_patients('tvae', CONTROL_PATIENT_IDS_TVAE),
], ignore_index=True)

control_slope_table = build_patient_slope_table(
    level1_control,
    patient_ids=list(set(CONTROL_PATIENT_IDS_CTGAN) | set(CONTROL_PATIENT_IDS_TVAE))
)
control_concordance_df = concordance_table(control_slope_table, stress_auc_df)

comparison = concordance_df.merge(
    control_concordance_df,
    on=['generator', 'label', 'model', 'metric'],
    suffixes=('_case_study', '_control')
)[['generator', 'label', 'model', 'metric',
   'concordance_rate_case_study', 'concordance_rate_control',
   'sign_test_pvalue_case_study', 'sign_test_pvalue_control']]

comparison